# INFO 5304 Project - Phase 1: Dataset Feasibility

**Team:** Tara Munjuluri, Sahithya Swaminathan, Colette D'Costa  
**GitHub Repository:** https://github.com/coletted1/CS-INFO-5304-Project.git


## Research Questions

This project studies country-level well-being and quality of life using the World Happiness Report. We focus on the following questions:

1. Which country-level factors (economic, social, health, and institutional) are most strongly associated with higher happiness scores?
2. Which countries or regions appear to outperform or underperform relative to their economic resources?
3. What are the major disparities in happiness-related predictors across countries?

## Raw Data Description

We use the official **World Happiness Report 2025 Figure 2.1 dataset** (publicly shared by the World Happiness Report team).

- Source page: https://www.worldhappiness.report/data-sharing/
- File: `WHR25_Data_Figure_2.1v3.xlsx`
- Unit of analysis: country-year observations (168 unique countries, 2011–2024)
- Core variables include life evaluation (happiness score) and explanatory factors such as GDP per capita, social support, healthy life expectancy, freedom, generosity, and perceptions of corruption.

**Important caveat:** the six predictor/decomposition columns (`Explained by: …`) are only populated for **2019–2024**. Rows for 2011–2018 contain ladder score and rank but have NaN for all six predictors. This is an artifact of the WHR's reporting methodology — decomposition estimates were not back-filled into earlier years in this file. The year 2013 is also absent (no WHR report was published that year).

This source is considered trustworthy because it is published by the World Happiness Report and based on Gallup World Poll methods with transparent documentation.

In [8]:
import re
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


In [ ]:
# Download source data from WHR with headers/fallbacks to avoid 403 hotlink errors

def download_bytes(url: str) -> bytes:
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=45) as response:
        return response.read()

source_url = None
errors = {}
raw_bytes = None

candidate_urls = [
    "https://files.worldhappiness.report/WHR25_Data_Figure_2.1v3.xlsx",
    "https://files.worldhappiness.report/WHR24_Data_Figure_2.1.xls",
]

for url in candidate_urls:
    try:
        raw_bytes = download_bytes(url)
        source_url = url
        break
    except Exception as exc:
        errors[url] = str(exc)

if raw_bytes is None:
    try:
        page = download_bytes("https://www.worldhappiness.report/data-sharing/").decode("utf-8", errors="ignore")
        match = re.search(r'https://files\.worldhappiness\.report/[^"]*WHR\d+_Data_Figure_2\.1[^"]*\.xlsx', page)
        if match:
            discovered_url = match.group(0)
            raw_bytes = download_bytes(discovered_url)
            source_url = discovered_url
    except Exception as exc:
        errors["data_sharing_page"] = str(exc)

if raw_bytes is None:
    raise RuntimeError(f"Could not download WHR data. Errors: {errors}")

# Ensure openpyxl is available for .xlsx reading
try:
    import openpyxl  # noqa: F401
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])

raw_df = pd.read_excel(BytesIO(raw_bytes))
print("Loaded from:", source_url)
print("Raw shape:", raw_df.shape)
raw_df.head(3)


Loaded from: https://files.worldhappiness.report/WHR25_Data_Figure_2.1v3.xlsx
Raw shape: (1969, 13)


,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,2024,147,Afghanistan,1.364,1.301,1.427,0.649,0.0,0.155,0.0,0.075,0.135,0.348
1,2023,143,Afghanistan,1.721,1.667,1.775,0.628,0.0,0.242,0.0,0.091,0.088,0.672
2,2022,137,Afghanistan,1.859,1.795,1.923,0.645,0.0,0.087,0.0,0.093,0.059,0.976


In [11]:
def normalize_col(col_name: str) -> str:
    col_name = str(col_name).strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name).strip("_")
    return col_name

df = raw_df.copy()
df.columns = [normalize_col(c) for c in df.columns]

print("Normalized columns:")
print(df.columns.tolist())


Normalized columns:
['year', 'rank', 'country_name', 'life_evaluation_3_year_average', 'lower_whisker', 'upper_whisker', 'explained_by_log_gdp_per_capita', 'explained_by_social_support', 'explained_by_healthy_life_expectancy', 'explained_by_freedom_to_make_life_choices', 'explained_by_generosity', 'explained_by_perceptions_of_corruption', 'dystopia_residual']


In [ ]:
def find_column(columns, required_keywords):
    """Find first column containing all required keywords."""
    for c in columns:
        if all(k in c for k in required_keywords):
            return c
    return None

cols = df.columns.tolist()

country_col = find_column(cols, ["country"]) or find_column(cols, ["country", "name"])
year_col = find_column(cols, ["year"])  # may not exist in this file
rank_col = find_column(cols, ["rank"])

# Try multiple possible names for happiness score across WHR file variants
score_col = (
    find_column(cols, ["ladder"])
    or find_column(cols, ["happiness", "score"])
    or find_column(cols, ["life", "evaluation"])
    or find_column(cols, ["life", "ladder"])
)

gdp_col = find_column(cols, ["gdp"])
social_col = find_column(cols, ["social"])
life_exp_col = find_column(cols, ["life", "expectancy"]) or find_column(cols, ["healthy", "life"])
freedom_col = find_column(cols, ["freedom"])
generosity_col = find_column(cols, ["generosity"])
corruption_col = find_column(cols, ["corruption"])

if country_col is None:
    raise ValueError(f"Could not find a country column. Available columns: {cols}")

selected_map = {
    country_col: "country",
    year_col: "year",
    rank_col: "rank",
    score_col: "ladder_score",
    gdp_col: "gdp_per_capita",
    social_col: "social_support",
    life_exp_col: "healthy_life_expectancy",
    freedom_col: "freedom_to_make_life_choices",
    generosity_col: "generosity",
    corruption_col: "perceptions_of_corruption",
}

# Keep only mappings where source column was found
selected_map = {k: v for k, v in selected_map.items() if k is not None}
analysis_df = df[list(selected_map.keys())].rename(columns=selected_map)

# Add report year if missing
if "year" not in analysis_df.columns:
    analysis_df["year"] = 2025

# Basic cleaning
analysis_df = analysis_df.dropna(subset=["country"]).copy()
analysis_df["country"] = analysis_df["country"].astype(str).str.strip()

numeric_cols = [c for c in analysis_df.columns if c != "country"]
for c in numeric_cols:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

sort_cols = [c for c in ["year", "ladder_score", "rank"] if c in analysis_df.columns]
if sort_cols:
    ascending = [True] + [False] * (len(sort_cols) - 1)
    analysis_df = analysis_df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
else:
    analysis_df = analysis_df.reset_index(drop=True)

print("Analysis-ready shape:", analysis_df.shape)
print("Final columns:", analysis_df.columns.tolist())
analysis_df.head(10)


Analysis-ready shape: (1969, 10)
Final columns: ['country', 'year', 'rank', 'ladder_score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom_to_make_life_choices', 'generosity', 'perceptions_of_corruption']


,country,year,rank,ladder_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom_to_make_life_choices,generosity,perceptions_of_corruption
0,Denmark,2011,1,7.856,NaN,NaN,NaN,NaN,NaN,NaN
1,Finland,2011,2,7.579,NaN,NaN,NaN,NaN,NaN,NaN
2,Norway,2011,3,7.524,NaN,NaN,NaN,NaN,NaN,NaN
3,Netherlands,2011,4,7.512,NaN,NaN,NaN,NaN,NaN,NaN
4,Switzerland,2011,6,7.499,NaN,NaN,NaN,NaN,NaN,NaN
5,Canada,2011,5,7.499,NaN,NaN,NaN,NaN,NaN,NaN
6,Sweden,2011,7,7.379,NaN,NaN,NaN,NaN,NaN,NaN
7,New Zealand,2011,8,7.372,NaN,NaN,NaN,NaN,NaN,NaN
8,Australia,2011,9,7.345,NaN,NaN,NaN,NaN,NaN,NaN
9,Ireland,2011,10,7.284,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Export analysis-ready dataset for Phase 1 submission
output_dir = Path("data/final")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "phase1_analysis_ready.csv"
analysis_df.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
analysis_df.describe(include="all").transpose().head(20)


Saved: C:\Users\sumasai\Downloads\INFO 5304\data\final\phase1_analysis_ready.csv


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,1969,168,Denmark,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1969.0,NaN,NaN,NaN,2017.714068,3.964913,2011.0,2015.0,2018.0,2021.0,2024.0
rank,1969.0,NaN,NaN,NaN,76.430168,43.942744,1.0,38.0,76.0,114.0,158.0
ladder_score,1969.0,NaN,NaN,NaN,5.451903,1.121864,1.364,4.596,5.4562,6.295,7.856
gdp_per_capita,872.0,NaN,NaN,NaN,1.220279,0.463448,0.0,0.901355,1.2635,1.567,2.209
social_support,872.0,NaN,NaN,NaN,1.078536,0.355057,0.0,0.85075,1.106699,1.361,1.84
healthy_life_expectancy,870.0,NaN,NaN,NaN,0.542915,0.222944,0.0,0.383,0.555,0.70475,1.137814
freedom_to_make_life_choices,871.0,NaN,NaN,NaN,0.56373,0.180204,0.0,0.450527,0.571,0.676,1.018
generosity,872.0,NaN,NaN,NaN,0.154261,0.086731,0.0,0.092,0.1405,0.205,0.569814
perceptions_of_corruption,871.0,NaN,NaN,NaN,0.144356,0.12029,0.0,0.062,0.113,0.18,0.587


## Plain-English Description of Code Steps

1. Imported Python libraries for data loading and cleaning.
2. Downloaded the World Happiness Report 2025 data file directly from the official source.
3. Standardized column names to make downstream processing consistent.
4. Identified relevant columns for our project (country, life evaluation, and key explanatory factors).
5. Cleaned values (trimmed country names, converted numeric fields, handled missing values).
6. Created a final analysis-ready dataframe and exported it to `data/final/phase1_analysis_ready.csv`.


## Analysis-Ready Dataframe Description

The final dataframe (`analysis_df`) is country-level data prepared for analysis.

- **Rows:** Each row corresponds to one country observation from the World Happiness Report 2025 dataset.
- **Columns (expected):**
  - `country`
  - `year`
  - `rank` (if available in source)
  - `ladder_score`
  - `gdp_per_capita`
  - `social_support`
  - `healthy_life_expectancy`
  - `freedom_to_make_life_choices`
  - `generosity`
  - `perceptions_of_corruption`

The exact row count and final column availability are printed in the code output, since some source files can differ slightly across report versions.

## Data Limitations / Issues Encountered

- This dataset is based on survey responses, so life evaluation is self-reported and can include subjective bias.
- Country-level aggregation may hide within-country inequalities.
- Variable definitions and naming can differ across report versions, requiring robust column matching.
- At this phase, we use one primary official file; later phases may require adding more datasets to support deeper causal or regional analysis.


## Questions for TA Reviewer

1. Is this level of data cleaning and feature preparation sufficient for Phase 1 expectations?
2. For future phases, do you recommend prioritizing regional controls and panel-style trends (multi-year merges) to strengthen analysis quality?
3. Would adding an OECD well-being dataset in Phase 2 be viewed as a meaningful extension relative to this current base dataset?
